In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import pymc as pm
import arviz as az

# ===============================
# 路径设置（沿用你原来的逻辑）
# ===============================
working_dir = Path.cwd().resolve()
root_dir = next((p for p in [working_dir, *working_dir.parents] if (p / "data").is_dir() and (p / "model").is_dir()), None)
if root_dir is None:
    raise RuntimeError("Run this notebook from within the repository.")
model_dir = root_dir / "model"
poisson_dir = model_dir / "poisson"
input_dir = root_dir / "data" / "processed" / "goal-filtered"

# 建议单独一个输出目录，避免和 MLE 混在一起
output_dir = poisson_dir / "lambdas" / "bayes-hierarchical-lambdas"

# PPS 输出目录
ppc_output_dir = poisson_dir / "posterior_predictive"
ppc_output_dir.mkdir(parents=True, exist_ok=True)

print("input_dir:", input_dir)
print("output_dir:", output_dir)
print("ppc_output_dir:", ppc_output_dir)

In [ ]:
# ============================
# 1. 赛季列表（你可以自己裁剪）
# ============================
season = {
    '1993-1994', '1994-1995', '1995-1996', '1996-1997', '1997-1998',
    # '1998-1999', '1999-2000', '2000-2001', '2001-2002', '2002-2003',
    # '2003-2004', '2004-2005', '2005-2006', '2006-2007', '2007-2008',
    # '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013',
    # '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018',
    # '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023',
    # '2023-2024'
}

# ============================
# 2. 修正版贝叶斯层级 Poisson 模型
#    - sum-to-zero 约束
#    - 更宽的 sigma 先验 (HalfStudentT)
#    - 全局 intercept 控制平均进球水平
#    - 更宽/更合理的主场优势先验
# ============================

def bayesian_team_parameters(
    year: str,
    draws: int = 2000,
    tune: int = 1000,
    target_accept: float = 0.9,
    random_seed: int = 123,
):
    """
    用修正版分层贝叶斯 Poisson + 主场优势模型，估计某赛季所有球队的攻防强度和 C。
    拟合后：
        - 保存 {year}-bayes-parameters.csv
        - 保存 {year}-trace.nc
    返回：
        result_df: 参数表
        trace:     PyMC InferenceData
    """

    # ---------- 数据 ----------
    input_file = input_dir / f"{year}.csv"
    if not input_file.exists():
        print(f"[WARN] {input_file} not found, skip.")
        return None, None

    data = pd.read_csv(input_file)

    teams = sorted(set(data["HomeTeam"]).union(data["AwayTeam"]))
    num_teams = len(teams)
    print(f"\n[Bayes] Fitting season {year} with {num_teams} teams, {len(data)} matches.")

    team_to_idx = {t: i for i, t in enumerate(teams)}

    home_idx = data["HomeTeam"].map(team_to_idx).values
    away_idx = data["AwayTeam"].map(team_to_idx).values
    home_goals = data["HomeGoal"].values
    away_goals = data["AwayGoal"].values

    # ---------- 如果 trace 已存在，直接加载 ----------
    trace_file = output_dir / f"{year}-trace.nc"
    param_file = output_dir / f"{year}-bayes-parameters.csv"

    if trace_file.exists() and param_file.exists():
        print(f"[Bayes] Trace & params already exist for {year}, loading...")
        trace = az.from_netcdf(trace_file)
        result_df = pd.read_csv(param_file)
        return result_df, trace

    # ---------- 构建模型 ----------
    with pm.Model() as model:
        # 全局 intercept，控制整体进球水平（log scale）
        intercept = pm.Normal("intercept", mu=0.0, sigma=1.0)

        # 攻防方差：用 HalfStudentT 允许更大的差异
        sigma_att = pm.HalfStudentT("sigma_att", nu=3, sigma=2.5)
        sigma_def = pm.HalfStudentT("sigma_def", nu=3, sigma=2.5)

        # 原始攻防参数（尚未 zero-sum + scale）
        alpha_att_raw = pm.Normal("alpha_att_raw", mu=0.0, sigma=1.0, shape=num_teams)
        alpha_def_raw = pm.Normal("alpha_def_raw", mu=0.0, sigma=1.0, shape=num_teams)

        # sum-to-zero 约束 + scale
        att_centered = alpha_att_raw - pm.math.mean(alpha_att_raw)
        def_centered = alpha_def_raw - pm.math.mean(alpha_def_raw)

        alpha_att = pm.Deterministic("alpha_att", att_centered * sigma_att)
        alpha_def = pm.Deterministic("alpha_def", def_centered * sigma_def)

        # 主场优势：更宽的先验（正态，中等偏正）
        C = pm.Normal("C", mu=0.15, sigma=0.3)

        # 线性预测子
        eta_home = intercept + alpha_att[home_idx] - alpha_def[away_idx] + C
        eta_away = intercept + alpha_att[away_idx] - alpha_def[home_idx]

        mu_home = pm.math.exp(eta_home)
        mu_away = pm.math.exp(eta_away)

        # 似然
        home_obs = pm.Poisson("home_goals_obs", mu=mu_home, observed=home_goals)
        away_obs = pm.Poisson("away_goals_obs", mu=mu_away, observed=away_goals)

        # 采样
        trace = pm.sample(
            draws=draws,
            tune=tune,
            target_accept=target_accept,
            chains=4,
            cores=4,
            random_seed=random_seed,
            progressbar=True,
        )

    # ---------- 保存 trace ----------
    print(f"[Bayes] Saving trace to {trace_file}")
    az.to_netcdf(trace, trace_file)

    # ---------- 后验摘要 ----------
    summary = az.summary(
        trace,
        var_names=["alpha_att", "alpha_def", "C", "intercept", "sigma_att", "sigma_def"],
        hdi_prob=0.95,
    )

    # 提取每队攻防均值与区间
    attack_mean, attack_low, attack_high = [], [], []
    defend_mean, defend_low, defend_high = [], [], []

    for i in range(num_teams):
        att_row = summary.loc[f"alpha_att[{i}]"]
        def_row = summary.loc[f"alpha_def[{i}]"]

        attack_mean.append(att_row["mean"])
        attack_low.append(att_row["hdi_2.5%"])
        attack_high.append(att_row["hdi_97.5%"])

        defend_mean.append(def_row["mean"])
        defend_low.append(def_row["hdi_2.5%"])
        defend_high.append(def_row["hdi_97.5%"])

    C_row = summary.loc["C"]

    result_df = pd.DataFrame({
        "Team": teams,
        "Attack_mean": attack_mean,
        "Attack_hdi_2.5%": attack_low,
        "Attack_hdi_97.5%": attack_high,
        "Defend_mean": defend_mean,
        "Defend_hdi_2.5%": defend_low,
        "Defend_hdi_97.5%": defend_high,
        "HomeAdvantage_mean": C_row["mean"],
        "HomeAdvantage_hdi_2.5%": C_row["hdi_2.5%"],
        "HomeAdvantage_hdi_97.5%": C_row["hdi_97.5%"],
    })

    result_df.to_csv(param_file, index=False)
    print(f"[Bayes] {param_file.name} saved.")

    return result_df, trace

# ============================
# 3. Posterior Predictive Simulation（修正版）
#    - 使用真正的 posterior samples
#    - 修复之前 sample 维度的 bug
# ============================

def posterior_predictive_simulation(
    year: str,
    num_sim_seasons: int = 1000,
    random_seed: int = 451,
):
    """
    对某个赛季进行 posterior predictive simulation。
    - 自动从 {year}-trace.nc 加载 trace
    - 返回 summary_df（积分分布 + 冠军 / Top4 / 降级概率）
      和 sim_points（num_sim_seasons x num_teams）
    """

    # --------- 读取数据 ---------
    input_file = input_dir / f"{year}.csv"
    data = pd.read_csv(input_file)

    teams = sorted(set(data["HomeTeam"]).union(data["AwayTeam"]))
    num_teams = len(teams)
    team_to_idx = {t: i for i, t in enumerate(teams)}

    home_idx = data["HomeTeam"].map(team_to_idx).values
    away_idx = data["AwayTeam"].map(team_to_idx).values
    num_matches = len(data)

    # --------- 加载 trace ---------
    trace_file = output_dir / f"{year}-trace.nc"
    if not trace_file.exists():
        raise FileNotFoundError(f"Trace file {trace_file} not found. Run bayesian_team_parameters first.")

    trace = az.from_netcdf(trace_file)

    # 将 (chain, draw) 合并成 sample 维度
    posterior = trace.posterior.stack(sample=("chain", "draw"))

    alpha_att = posterior["alpha_att"].values  # (sample, num_teams)
    alpha_def = posterior["alpha_def"].values  # (sample, num_teams)
    C_samples = posterior["C"].values         # (sample,)
    intercept_samples = posterior["intercept"].values  # (sample,)

    num_samples = alpha_att.shape[0]

    print(f"\n[PPS] Season {year}: {num_teams} teams, {num_matches} matches.")
    print(f"[PPS] Using {num_samples} posterior samples, simulating {num_sim_seasons} seasons.")

    rng = np.random.default_rng(random_seed)

    # 每次模拟的积分结果
    sim_points = np.zeros((num_sim_seasons, num_teams), dtype=int)

    # --------- 逐赛季模拟 ---------
    for s in range(num_sim_seasons):
        # 随机选一个 posterior 样本
        idx = rng.integers(0, num_samples)

        att = alpha_att[idx]        # (num_teams,)
        deff = alpha_def[idx]
        C = C_samples[idx]
        intercept = intercept_samples[idx]

        points = np.zeros(num_teams, dtype=int)

        # 每一场比赛
        for m in range(num_matches):
            h = home_idx[m]
            a = away_idx[m]

            eta_home = intercept + att[h] - deff[a] + C
            eta_away = intercept + att[a] - deff[h]

            lam_h = np.exp(eta_home)
            lam_a = np.exp(eta_away)

            hg = rng.poisson(lam_h)
            ag = rng.poisson(lam_a)

            if hg > ag:
                points[h] += 3
            elif ag > hg:
                points[a] += 3
            else:
                points[h] += 1
                points[a] += 1

        sim_points[s, :] = points

    # --------- 汇总统计 ---------
    mean_pts = sim_points.mean(axis=0)
    median_pts = np.median(sim_points, axis=0)
    sd_pts = sim_points.std(axis=0)
    q025 = np.quantile(sim_points, 0.025, axis=0)
    q975 = np.quantile(sim_points, 0.975, axis=0)

    # 冠军概率
    champions = np.argmax(sim_points, axis=1)  # 每个赛季积分最高队的 index
    champion_prob = np.array([
        np.mean(champions == i) for i in range(num_teams)
    ])

    # Top 4 概率
    # argsort 从小到大，所以取 [:, -4:] 是前四高积分
    top4_prob = np.zeros(num_teams)
    bottom3_prob = np.zeros(num_teams)

    sorted_indices = np.argsort(sim_points, axis=1)
    top4 = sorted_indices[:, -4:]
    bottom3 = sorted_indices[:, :3]

    for i in range(num_teams):
        top4_prob[i] = np.mean(np.any(top4 == i, axis=1))
        bottom3_prob[i] = np.mean(np.any(bottom3 == i, axis=1))

    summary_df = pd.DataFrame({
        "Team": teams,
        "Points_mean": mean_pts,
        "Points_median": median_pts,
        "Points_sd": sd_pts,
        "Points_2.5%": q025,
        "Points_97.5%": q975,
        "Champion_prob": champion_prob,
        "Top4_prob": top4_prob,
        "Relegation_prob": bottom3_prob,
    })

    # 按平均积分排序
    summary_df = summary_df.sort_values("Points_mean", ascending=False).reset_index(drop=True)

    # 保存结果
    ppc_file = ppc_output_dir / f"{year}-posterior_predictive_table.csv"
    summary_df.to_csv(ppc_file, index=False)
    print(f"[PPS] Saved PPS table to {ppc_file}")

    return summary_df, sim_points


In [ ]:
year = "1993-1994"
params_9394, trace_9394 = bayesian_team_parameters(year)


In [ ]:
summary_9394, sim_points_9394 = posterior_predictive_simulation("1993-1994", num_sim_seasons=1000)
summary_9394


In [ ]:
# 赛季列表（沿用你原来的）
season = {
    '1993-1994', '1994-1995', '1995-1996', '1996-1997', '1997-1998',
    '1998-1999', '1999-2000', '2000-2001', '2001-2002', '2002-2003',
    '2003-2004', '2004-2005', '2005-2006', '2006-2007', '2007-2008',
    # '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013',
    # '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018',
    # '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023',
    # '2023-2024'
}


def bayesian_team_parameters(
    year: str,
    draws: int = 2000,
    tune: int = 1000,
    target_accept: float = 0.9,
    random_seed: int = 123
):
    """
    用分层贝叶斯 Poisson + 主场优势模型，估计某一赛季所有球队的攻防强度
    以及联盟主场优势 C。

    输出：
        result_df: 每队 Attack/Defend 的后验均值和 95% CI 以及 HomeAdvantage 均值
        trace: PyMC 采样结果 (InferenceData)
    """

    # ========== 读取数据 ==========
    input_file = input_dir / f"{year}.csv"
    if not input_file.exists():
        print(f"Warning: {input_file} does not exist. Skipped.")
        return None, None

    data = pd.read_csv(input_file)

    teams = sorted(set(data['HomeTeam']).union(data['AwayTeam']))
    num_teams = len(teams)
    print(f"Processing {year} with {num_teams} teams (Bayesian).")

    team_to_idx = {team: i for i, team in enumerate(teams)}

    home_idx = data['HomeTeam'].map(team_to_idx).values
    away_idx = data['AwayTeam'].map(team_to_idx).values

    home_goals = data['HomeGoal'].values
    away_goals = data['AwayGoal'].values

    # ========== 检查 trace 文件是否已经存在 ==========
    output_dir.mkdir(parents=True, exist_ok=True)
    trace_file = output_dir / f"{year}-trace.nc"

    if trace_file.exists():
        print(f"Trace for {year} already exists. Loading from file.")
        trace = az.from_netcdf(trace_file)

        # 同时加载参数 CSV
        param_file = output_dir / f"{year}-bayes-parameters.csv"
        result_df = pd.read_csv(param_file)

        return result_df, trace

    # ========== 构建 PyMC 模型 ==========
    with pm.Model() as model:

        mu_att = pm.Normal("mu_att", mu=0.0, sigma=1.0)
        mu_def = pm.Normal("mu_def", mu=0.0, sigma=1.0)

        sigma_att = pm.HalfNormal("sigma_att", sigma=1.0)
        sigma_def = pm.HalfNormal("sigma_def", sigma=1.0)

        alpha_att = pm.Normal("alpha_att", mu=mu_att, sigma=sigma_att, shape=num_teams)
        alpha_def = pm.Normal("alpha_def", mu=mu_def, sigma=sigma_def, shape=num_teams)

        C = pm.Normal("C", mu=0.3, sigma=0.2)

        eta_home = alpha_att[home_idx] - alpha_def[away_idx] + C
        eta_away = alpha_att[away_idx] - alpha_def[home_idx]

        lambda_home = pm.Deterministic("lambda_home", pm.math.exp(eta_home))
        lambda_away = pm.Deterministic("lambda_away", pm.math.exp(eta_away))

        home_obs = pm.Poisson("home_goals_obs", mu=lambda_home, observed=home_goals)
        away_obs = pm.Poisson("away_goals_obs", mu=lambda_away, observed=away_goals)

        trace = pm.sample(
            draws=draws,
            tune=tune,
            target_accept=target_accept,
            random_seed=random_seed,
            chains=4,
            cores=4,
            progressbar=True
        )

    # ========== 保存 trace ==========
    print(f"Saving trace for {year} to {trace_file} ...")
    az.to_netcdf(trace, trace_file)
    print("Trace saved successfully!")

    # ========== 后验摘要 ==========
    summary = az.summary(
        trace,
        var_names=["alpha_att", "alpha_def", "C"],
        hdi_prob=0.95
    )

    attack_means, attack_low, attack_high = [], [], []
    defend_means, defend_low, defend_high = [], [], []

    for i in range(num_teams):
        att = summary.loc[f"alpha_att[{i}]"]
        dff = summary.loc[f"alpha_def[{i}]"]

        attack_means.append(att["mean"])
        attack_low.append(att["hdi_2.5%"])
        attack_high.append(att["hdi_97.5%"])

        defend_means.append(dff["mean"])
        defend_low.append(dff["hdi_2.5%"])
        defend_high.append(dff["hdi_97.5%"])

    C_row = summary.loc["C"]

    result_df = pd.DataFrame({
        "Team": teams,
        "Attack_mean": attack_means,
        "Attack_hdi_2.5%": attack_low,
        "Attack_hdi_97.5%": attack_high,
        "Defend_mean": defend_means,
        "Defend_hdi_2.5%": defend_low,
        "Defend_hdi_97.5%": defend_high,
        "HomeAdvantage_mean": C_row["mean"],
        "HomeAdvantage_hdi_2.5%": C_row["hdi_2.5%"],
        "HomeAdvantage_hdi_97.5%": C_row["hdi_97.5%"],
    })

    param_file = output_dir / f"{year}-bayes-parameters.csv"
    result_df.to_csv(param_file, index=False)

    print(result_df.head())
    print(f"{param_file.name} saved successfully!\n")

    return result_df, trace



if __name__ == "__main__":
    for year in sorted(season):
        print(f"\n=== Bayesian estimation for {year} ===")
        result_df, trace = bayesian_team_parameters(year)

    print("All seasons processed (Bayesian).")
